In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from kret_studies import *
from kret_studies.notebook import *
from kret_studies.complex import *
from kret_studies.openai_gym import *

logger = get_notebook_logger()

Loaded environment variables from /Users/Akseldkw/Desktop/Columbia/ORCS4529/.env.


INFO:datasets:JAX version 0.7.2 available.


/Users/Akseldkw/coding/kretsinger/data/nb_log.log


In [3]:
env_vars = os.environ.copy()
wand_db_dir = env_vars["OUTPUT_DIR"] + "/wandb"

## Delete above

In [4]:
from gymnasium.wrappers.common import TimeLimit
from gymnasium.envs.classic_control.cartpole import CartPoleEnv
from gymnasium.spaces import Discrete, Box
import typing as t
from collections import deque
from numpy.typing import NDArray
import random

In [5]:
import torch 

from collections import deque

import numpy as np

%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt

# import gym

In [6]:
# %pip install wandb
import wandb

wandb.login()
run = wandb.init(dir=wand_db_dir, project="orcs4529_hw3_policy_gradient")

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have 

## Policy Gradient

In the previous lab, we talked about value based method for reinforcement learning. In this lab, we focus on policy based method.

In policy based methods, intead of defining a value function $Q_\theta(s,a)$ and inducing a policy based on argmax, we parameterize a stochastic policy directly. The policy is parameterized as a categorical distribution over actions. Let it be $\pi_\phi(s)$ with parameter $\phi$, then the policy is defined by sampling actions $$a \sim \pi_\phi(s)$$

The policy induces a probability $p(\tau)$ over trajectories $\tau = \{s_0,a_0,s_1,a_1,..\}$. The expected total discounted reward is

$$\rho(\phi) = \mathbb{E}_{\tau \sim p(\tau)} \big[R(\tau)\big] = \mathbb{E}_{\pi_\phi} \big[\sum_{t=0}^\infty r_t \gamma^t \big]$$

The aim is to find $\phi$ such that the expected reward induced by $\pi_\phi$ is maximized.

### Policy Gradient Computation

We can derive policy gradient

$$\nabla_\phi \rho(\phi) = \mathbb{E}_{\pi} \big[\sum_{t=0}^\infty Q^{\pi_\phi}(s_t,a_t) \nabla_\phi \log \pi_\phi(s_t, a_t) \big]$$

To compute the gradient for update $\phi \leftarrow \phi + \alpha \nabla_\phi \rho(\phi)$, we need to estimate $Q^{\pi_\phi}(s,a)$. Since $Q^{\pi_\phi}(s,a)$ is usually not analytically accessible, it can be approximated by
1. Monte Carlo estimate
2. Train a value function $Q_\theta(s,a) \approx Q^{\pi_\phi}(s,a)$ and use it as a proxy
3. Mixture of both above

Before estimating $Q^{\pi_\phi}(s,a)$, let us write a parameterized policy over actions. The policy $\pi_\phi(s)$ takes a state as input and outputs a categorical distribution over actions. For example, if we have two actions, the probability vector to output is of the form $\pi_\phi(s)=[0.6,0.4]$.

**Loss function**
Given samples of state action pairs $(s_i, a_i)$ and estimate $Q_i$ for $Q^{\pi_\phi}(s_i, a_i)$, for $i=1,\ldots, $ we  set the loss function  as
$$-\frac{1}{N} \sum_{i=1}^N Q_i \log(\pi_\phi(s_i, a_i)) $$
The loss function enables us  to compute policy gradients in implementation. The negative gradient of the above loss function has the form

$$\frac{1}{N} \sum_{i=1}^N Q_i  \nabla_\phi \log \pi_\phi(s_i, a_i) $$

where $Q_i$ are estimated and $\nabla_\phi \log\pi_\phi(s_i, a_i)$ are computed via backprop.

In [7]:
import policy_nn as policy_nn
from policy_nn import Policy

Try to rollout trajecories using the policy

In [8]:
envname = "CartPole-v0"  # environment name
max_traj_steps = 1_000  # max number of steps per episode

env = t.cast(CartPoleEnv, gym.make(envname, max_episode_steps=max_traj_steps))
obssize = t.cast(Box, env.observation_space).low.size
actsize = t.cast(Discrete, env.action_space).n
policyinit = Policy(obssize, actsize, 0.1)

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


In [9]:
# Below is a set of template code for running a policy to interact with the environment
# It initializes a policy and runs it
# Note that you may not be able to run the code properly if there still some undefined components on the Policy class


obs, info = env.reset()
done = False
while not done:
    prob = policyinit.compute_prob(np.expand_dims(obs, 0))
    prob /= np.sum(prob)  # normalizing again to account for numerical errors
    action = np.random.choice(actsize, p=prob.flatten(), size=1).item()
    obs, reward, done, term, info = env.step(action)

### Estimate $Q^\pi(s,a)$

To estimate $Q^\pi(s,a)$, we can rollout the policy until the episode ends and do monte carlo estimate. In particular, under policy $\pi$, we start from state action $(s_0,a_0)$ and rollout the policy to generate a trajectory $\{s_0,a_0,s_1,a_1...s_T,a_T\}$, with corresponding reward $r_0,r_1...r_T$. Monte carlo estimate is

$$\hat{Q}_{MC}(s,a) = \sum_{t=0}^T r_t \gamma^t \approx Q^\pi(s,a)$$

This estimate by itself is of high variance. Using pure monte carlo estimate may work but the gradient can have large variance and hence take the algorithm  a long time to converge. We can reduce variance using baseline. Recall the derivation of PG

$$\nabla_\phi \rho(\phi) = \mathbb{E}_{\pi_\phi} \big[\sum_{t=0}^\infty Q^{\pi_\phi}(s_t,a_t) \nabla_\phi \log \pi_\phi(s_t, a_t) \big] = \mathbb{E}_{\pi_\phi} \big[\sum_{t=0}^\infty ( Q^{\pi_\phi}(s_t,a_t) - b(s_t)) \nabla_\phi \log \pi_\phi(s_t, a_t) \big]$$

where $b(s_t)$ can be any function of state $s_t$. $b(s_t)$ is called baseline. Optimal baseline function is hard to compute, but a good proxy is the value function $V^{\pi_\phi}(s_t)$. Hence the gradient has the form
$$\nabla_\phi \rho(\pi_{\phi}) = \mathbb{E}_{\pi} \big[\sum_{t=0}^\infty A^{\pi_\phi}(s_t,a_t) \nabla_\phi \log \pi_\phi(s_t, a_t) \big]$$

where $A^{\pi_\phi}(s,a)$ is the advantage. Hence we can train a value function $V^{\pi_\phi}(s)$ along side the policy and use it as baseline to reduce the variance of PG.

Hence we also parameterize a value function $V_\theta(s) \approx V^{\pi_\phi}(s)$ with parameter $\theta$ to serve as baseline. The function takes as input the states $s$ and outputs a real value.

Notice that unlike DQN, where $Q_\theta(s,a) \approx Q^\ast(s,a)$, now we have $V_\theta(s) \approx V^{\pi_\phi}(s)$. Therefore, we have a moving target to approximate, that changes with the current policy $\pi_\phi$. As $\phi$ is updated by PG, $\pi_\phi$ keeps changing, which changes $V^{\pi_\phi}(s)$ as well. We need to adapt $V_\theta(s)$ online to cater for the change in policy.

Recall that to evaluate a policy $\pi$, we collect rollouts using $\pi$. If we start with state $s_0$, the reward following $\pi$ thereafter is $r_0...r_{T}$  then

$$V^\pi(s_0) \approx \sum_{t=0}^{T} r_t \gamma^{t} = \hat{V}(s_0)$$

In general, given a trajectory $(s_0, a_0, s_1, a_1, r_1, s_2, a_2, r_2, ..., s_{T+1})$

$$\hat{V}(s_i) = \sum_{i=t}^{T} r_i \gamma^{i-t}$$

And the objective to minimize over is
$$\frac{1}{T+1} \sum_{i=0}^{T} (V_\theta(s_i) - \hat{V}(s_i))^2$$

Since the policy keeps updating, we do not need to minimize the above objective to optimality. In practice, taking one gradient step w.r.t. above objective suffices.

In the code cell below, define the neural network to learn value function estimate. The implementation is similar to Qfunction class in lab3, except that inputs are only states, and not actions.

In [10]:
import value_nn as value_nn
from value_nn import ValueFunction

### Summary of pseudocode

The critical components of the pseudocode are as follows.

**Collect trajectories** Given current policy $\pi_\phi$, we can rollout using the policy by executing $a_t \sim \pi_\phi(s_t)$.

**Update value function** Value function update is based on minimizing the L2 loss between predicted value function and estimated value functions. For each state $s_i, i=0,\ldots, T$ in a trajectory of length $T+1$, compute $\hat{V}(s_i)$ as dicounted reward over the rest of the path (as defined above).  
Then take one gradient step to update $\theta$ using the gradient of the following loss:

$$\frac{1}{T+1} \sum_{i=0}^T(V_\theta(s_i) - \hat{V}(s_i))^2$$

For your conveience, below we have provided a function discounted_rewards(r,gamma) that takes as inputs a list of $T$ rewards $r$ and computes all discounted rewards $\hat V(s_i), i=0, 1, 2, \ldots, T$.

**Update policy using PG** To compute PG, we need to first monte carlo estimate action-value function $\hat{Q}(s_i,a_i)$. Given a trajectory with rewards $r=[r_0, r_1, r_2, \ldots, r_T]$, this can also be computed for all $s_i, a_i$ in this trajectory using the discounted_rewards(r, gamma) function below.

Then use value function as a baseline to compute advantage

$$\hat{A}(s_i,a_i) = \hat{Q}(s_i,a_i) - V_\theta(s_i)$$

Then compute surrogate loss

$$L = - \frac{1}{(T+1)}\sum_{i} \hat{A}(s_i,a_i) \log \pi(a_i|s_i) $$

The policy is updated by $$\phi \leftarrow \phi - \alpha  \nabla_\phi L \approx \phi + \alpha \nabla_\phi \rho(\pi_\phi)$$

In [11]:
import training_loop as training_loop
from training_loop import discounted_rewards, train_policy_gradient

### Main implementation : Policy gradient algorithm

Combine all the above steps and implement the policy gradient algorithm with value function baseline in the cell below. The use of baseline is optional.

In [12]:
# %%wandb
# remove the above line if you do not want to see wandb plots in your notebook. You can always see them on the wandb website.

# You can change the code in this cell anyway you want
# However, just make sure per epsiode reward during the run of this algorithm is being recorded in list rrecord
# and logged on wandb as in the last few lines.

# parameter initializations (you can change any of these)
alpha = 1e-2  # learning rate for PG
beta = 1e-3  # learning rate for baseline
numtrajs = 5  # num of trajecories from the current policy to collect in each iteration
iterations = 300  # total num of iterations
# envname = "CartPole-v0"  # environment name
gamma = 0.99  # discount

# initialize networks
actor = Policy(obssize, actsize, alpha)  # policy initialization: IMPORTANT: this is the policy you will be scored on
baseline = ValueFunction(obssize, beta)  # baseline initialization

# To record training reward for logging and plotting purposes
rrecord = []

In [13]:
# main iteration TODO Re-Add
train_policy_gradient(actor, baseline, env, rrecord, max_traj_steps)

100%|██████████| 300/300 [01:32<00:00,  3.25it/s]


Finally, we evaluate the performance of the trained agent. We will evaluate the performance of the trained policy. The evaluation will be run for 100 epsiodes and print out the average performance across these episodes. Please **do not** change the code below.

In [16]:
# DO NOT CHANGE CODE HERE


### DO NOT CHANGE
def evaluate(policy: Policy, env: CartPoleEnv, episodes: int):
    # main iteration
    score = 0
    for episode in tqdm.tqdm(range(episodes)):

        obs, info = env.reset()
        done = False
        rsum = 0

        while not done:

            p = policy.compute_prob(np.expand_dims(obs, 0)).ravel()
            p /= np.sum(p)
            action = np.random.choice(np.arange(2), size=1, p=p).item()

            # env stepping forward
            newobs, r, done, _, info = env.step(action)

            # update data
            rsum += r
            obs = newobs

        wandb.log({"eval reward": rsum})
        score += rsum
    score = score / episodes

    return score

In [17]:
# DO NOT CHANGE CODE HERE
# after training, we will evaluate the performance of the learned policy "actor"
# on a target environment
env_test = gym.make(envname)
eval_episodes = 1000
score = evaluate(actor, env_test, eval_episodes)
wandb.run.summary["score"] = score
print("eval performance of the learned policy: {}".format(score))

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(
100%|██████████| 1000/1000 [01:38<00:00, 10.16it/s]

eval performance of the learned policy: 1497.516


In [18]:
run.finish()

eval reward,▄▃▃▂▄▅▂▃▁▁▃▃█▄▅▂▂▂▃▃▅▃▁▆▂▇▃▃▂▄▃▁▃▂▄▂▄▅▇▂
training reward,▁▁▁▁▁▂▃██▇▅▂▂▂▂██▆█████████████████████▆
training reward moving average,▁▁▁▁▁▂▃▆▄▄▃▄▅▇▇▇▇▇▇█████████████████████
eval reward,2189
score,1497.516
training reward,761
training reward moving average,974.41414


In [19]:
score

1497.516